In [2]:
# cross sections at an angle through a fixed center point

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import wrf
import xarray
from netCDF4 import Dataset
from datetime import timedelta, date

import cartopy.crs as crs
from cartopy.feature import NaturalEarthFeature
from cartopy import crs
from cartopy.feature import NaturalEarthFeature, COLORS
from matplotlib.cm import get_cmap

In [3]:
# colormap definition with white in the middle
n=39
x = 0.5
lower = plt.cm.seismic_r(np.linspace(0, x, n))
white = plt.cm.seismic_r(np.ones(80-2*n)*x)
upper = plt.cm.seismic_r(np.linspace(1-x, 1, n))
colors = np.vstack((lower, white, upper))
from matplotlib.colors import LinearSegmentedColormap
tmap = LinearSegmentedColormap.from_list('map_white', colors)

In [11]:
########## angled cross section through a fixed center point #########

# ── WRF data location ──────────────────────────────────────────────────────────
baseFolder = '/glade/derecho/scratch/gduine/mountain_fire/111m/ifire2/'

# Glob all domain-3 output files in chronological order
wrf_files = sorted(glob.glob(os.path.join(baseFolder, 'wrfout_d04_*')))
print(f'Found {len(wrf_files)} WRF files')

# ── Cross-section geometry ──────────────────────────────────────────────────────
center_lat       = 34.318    # center point latitude  (°N)
center_lon       = -118.968  # center point longitude (°W, negative)
angle_from_north = 35.0      # degrees clockwise from north
half_length_km   = 10.0      # half-length of the cross section (km)

# Compute start (SW) and end (NE) lat/lon
angle_rad  = np.radians(angle_from_north)
delta_lat  = half_length_km * np.cos(angle_rad) / 111.0
delta_lon  = half_length_km * np.sin(angle_rad) / (111.0 * np.cos(np.radians(center_lat)))

start_lat = center_lat - delta_lat
start_lon = center_lon - delta_lon
end_lat   = center_lat + delta_lat
end_lon   = center_lon + delta_lon

print(f'Cross section: center ({center_lat}°N, {center_lon}°W), angle {angle_from_north}° from N')
print(f'  SW end: ({start_lat:.4f}°N, {start_lon:.4f}°W)')
print(f'  NE end: ({end_lat:.4f}°N,  {end_lon:.4f}°W)')
print(f'  Along-section wind = U·sin({angle_from_north:.0f}°) + V·cos({angle_from_north:.0f}°)')
print(f'  Positive = toward NE end of section')

# ── Plot / axis settings ────────────────────────────────────────────────────────
g          = 9.81
Vs_levels  = np.arange(-24, 26,  2)   # along-section wind contour levels
W_levels   = np.arange(-5,  5.2, 0.2)
th_levels  = np.arange(260, 340, 1)

xlim1 = -half_length_km   # km from center, SW end
xlim2 =  half_length_km   # km from center, NE end
ylim1 = 0
ylim2 = 3000

# Froude search: only look south of this distance from center (km)
north_limit_km = 10.0

# ── One-time setup (runs on first file) ────────────────────────────────────────
ter        = None
xy         = None
ter_line   = None
dist_1d    = None
cross_start= None
cross_end  = None

# ── Main loop: files then timesteps ────────────────────────────────────────────
for fName in wrf_files:

    ncfile = Dataset(fName)
    print(f'\nOpened: {fName}')

    # First-file initialisation: terrain + cross-section path
    if ter is None:
        ter = wrf.getvar(ncfile, 'ter', -1)

        cross_start = wrf.CoordPair(lat=start_lat, lon=start_lon)
        cross_end   = wrf.CoordPair(lat=end_lat,   lon=end_lon)
        ter_line    = wrf.interpline(ter, wrfin=ncfile,
                                     start_point=cross_start, end_point=cross_end)

        latLocs = [start_lat, end_lat]
        lonLocs = [start_lon, end_lon]
        xy = wrf.ll_to_xy(ncfile, latLocs, lonLocs)

        nx_cross = len(ter_line)
        dist_1d  = np.linspace(xlim1, xlim2, nx_cross)   # km from center
        print(f'Cross-section has {nx_cross} grid points')

    # 6 timesteps per file (every 10 minutes within each hour)
    for idt in range(1):

        print(f'  timestep {idt}')

        # Read 3-D fields
        timeWRF = wrf.getvar(ncfile, 'Times',   idt)
        z       = wrf.getvar(ncfile, 'z',       idt)
        U       = wrf.getvar(ncfile, 'ua',      idt)   # zonal wind (destaggered)
        V       = wrf.getvar(ncfile, 'va',      idt)   # meridional wind (destaggered)
        W       = wrf.getvar(ncfile, 'wa',      idt)
        th      = wrf.getvar(ncfile, 'th',      idt)
        pblh    = wrf.getvar(ncfile, 'PBLH',    idt)
        qv      = wrf.getvar(ncfile, 'QVAPOR',  idt)
        thv     = th * (1.0 + 0.61 * qv)

        # Time string
        ts               = pd.to_datetime(wrf.to_np(timeWRF))
        tsPST            = ts - pd.DateOffset(hours=8)
        tWRFstr          = ts.strftime('%Y%m%d_%H%M')
        tWRF_strPST      = tsPST.strftime('%Y%m%d %H%M')
        tWRF_strPST_fName= tsPST.strftime('%Y%m%d_%H%M')

        # Cross-section interpolation along the angled path
        xy_line    = wrf.xy(V,   start_point=xy[:, 0], end_point=xy[:, 1])
        U_cross    = wrf.interp2dxy(U,   xy_line, meta=True)
        V_cross    = wrf.interp2dxy(V,   xy_line, meta=True)
        W_cross    = wrf.interp2dxy(W,   xy_line, meta=True)
        z_cross    = wrf.interp2dxy(z,   xy_line, meta=True)
        th_cross   = wrf.interp2dxy(th,  xy_line, meta=True)
        thv_cross  = wrf.interp2dxy(thv, xy_line, meta=True)
        pblh_cross = wrf.interpline(pblh, wrfin=ncfile,
                                    start_point=cross_start, end_point=cross_end)

        # Rotate wind to along-section component
        # Along-section (positive = toward NE end):  Vs =  U·sin(θ) + V·cos(θ)
        U_np      = wrf.to_np(U_cross)                                    # (nz, nx)
        V_np      = wrf.to_np(V_cross)
        Valong_np = U_np * np.sin(angle_rad) + V_np * np.cos(angle_rad)

        # 2-D distance array for contourf (same shape as V_cross)
        nz, nx = U_np.shape
        dist_2d = np.tile(dist_1d, (nz, 1))

        # Numpy arrays for Froude number calculation (use along-section wind)
        pblh_np = wrf.to_np(pblh_cross)
        thv_np  = wrf.to_np(thv_cross)
        z_np    = wrf.to_np(z_cross)

        if   nz == 54:  offset = 2
        elif nz == 80:  offset = 3
        elif nz == 106: offset = 4
        else:           offset = 2

        fr_profile = np.full(nx, np.nan)

        for ix in range(nx):
            z_col      = z_np[:, ix]
            thv_col    = thv_np[:, ix]
            Valong_col = Valong_np[:, ix]
            zi         = pblh_np[ix]
            if zi <= 0:
                zi = 1.0
            above = np.where(z_col > zi)[0]
            if len(above) == 0:
                continue
            ind_zi   = max(above[0] - 1, 0)
            mean_thv = np.nanmean(thv_col[:ind_zi + 1])
            ind_free = min(ind_zi + offset, nz - 1)
            thv_free = thv_col[ind_free]
            mean_Vs  = np.nanmean(Valong_col[:ind_zi + 1])
            g_prime  = g * (thv_free - mean_thv) / mean_thv
            if g_prime <= 0:
                continue
            fr_profile[ix] = np.abs(mean_Vs) / np.sqrt(g_prime * zi)

        # ── Froude transition detection (in distance space) ─────────────────────
        is_super = fr_profile > 1.0
        transition_dists = {'sub_to_super': None, 'super_to_sub': None}

        for ix in range(nx - 2, -1, -1):   # NE to SW
            if np.isnan(fr_profile[ix]) or np.isnan(fr_profile[ix + 1]):
                continue
            if dist_1d[ix] > north_limit_km:
                continue
            fr_a, fr_b     = fr_profile[ix], fr_profile[ix + 1]
            dist_a, dist_b = dist_1d[ix],    dist_1d[ix + 1]
            dist_cross = dist_a + (1.0 - fr_a) * (dist_b - dist_a) / (fr_b - fr_a)

            if is_super[ix] and not is_super[ix + 1]:
                if transition_dists['sub_to_super'] is None:
                    transition_dists['sub_to_super'] = dist_cross
            elif not is_super[ix] and is_super[ix + 1]:
                if transition_dists['sub_to_super'] is not None \
                        and transition_dists['super_to_sub'] is None:
                    transition_dists['super_to_sub'] = dist_cross

        # ── Plotting ────────────────────────────────────────────────────────────
        fig = plt.figure(figsize=(22, 7))
        xlabel_str = f'Distance along {angle_from_north:.0f}° cross section [km]'
        time_label = 'WRF 1 km ' + tWRF_strPST[:8] + ' ' + tWRF_strPST[9:13] + ' PST'

        # ── Left panel: along-section wind ─────────────────────────────────────
        ax_V = plt.subplot(1, 2, 1)
        Vs_contours = ax_V.contourf(dist_2d, wrf.to_np(z_cross), Valong_np,
                                    levels=Vs_levels, cmap=tmap, extend='both')
        th_contours = ax_V.contour(dist_2d, wrf.to_np(z_cross), wrf.to_np(th_cross),
                                   levels=th_levels, colors='k')
        ax_V.fill_between(dist_1d, np.zeros(nx_cross), wrf.to_np(ter_line),
                          facecolor='saddlebrown')
        plt.clabel(th_contours, inline=1, fontsize=20, levels=th_levels[::3], fmt='%1.0f')
        cbar_V = fig.colorbar(Vs_contours, ax=ax_V, extend='both',
                              orientation='horizontal', location='bottom',
                              shrink=0.7, pad=0.15, aspect=30)
        cbar_V.set_label('Along-section wind [m s$^{-1}$]', fontsize=16)
        cbar_V.ax.tick_params(labelsize=14)
        ax_V.scatter([0], [50], marker='^', color='yellow', edgecolor='black',
                     linewidth=2.0, s=500, zorder=8)
        ax_V.set_ylabel('Elevation [m]', fontsize=24)
        ax_V.set_xlabel(xlabel_str, fontsize=20)
        ax_V.set_title(f'Along-section wind (U·sin{angle_from_north:.0f}°+V·cos{angle_from_north:.0f}°) [m s$^{{-1}}$]',
                       fontsize=18)
        plt.xticks(fontsize=20)
        plt.yticks(fontsize=20)
        plt.text(xlim1 + 1, ylim2 - 200, time_label,
                 fontsize=20,
                 bbox=dict(facecolor='white', edgecolor='white', boxstyle='round,pad=0.3'),
                 zorder=10)
        if transition_dists['sub_to_super'] is not None:
            ax_V.axvline(x=transition_dists['sub_to_super'], color='cyan', linewidth=3.5,
                         linestyle='--', label='sub to super', zorder=9)
        if transition_dists['super_to_sub'] is not None:
            ax_V.axvline(x=transition_dists['super_to_sub'], color='orange', linewidth=3.5,
                         linestyle='--', label='super to sub', zorder=9)
        if any(transition_dists.values()):
            ax_V.legend(fontsize=16, loc='upper right')
        plt.ylim(ylim1, ylim2)
        plt.xlim(xlim1, xlim2)

        # ── Right panel: W-component ────────────────────────────────────────────
        ax_W = plt.subplot(1, 2, 2)
        W_contours = ax_W.contourf(dist_2d, wrf.to_np(z_cross), wrf.to_np(W_cross),
                                   levels=W_levels, cmap=tmap, extend='both')
        th_contours = ax_W.contour(dist_2d, wrf.to_np(z_cross), wrf.to_np(th_cross),
                                   levels=th_levels, colors='k')
        ax_W.fill_between(dist_1d, np.zeros(nx_cross), wrf.to_np(ter_line),
                          facecolor='saddlebrown')
        plt.clabel(th_contours, inline=1, fontsize=20, levels=th_levels[::3], fmt='%1.0f')
        cbar_W = fig.colorbar(W_contours, ax=ax_W, extend='both',
                              orientation='horizontal', location='bottom',
                              shrink=0.7, pad=0.15, aspect=30)
        cbar_W.set_label('W [m s$^{-1}$]', fontsize=16)
        cbar_W.ax.tick_params(labelsize=14)
        ax_W.scatter([0], [50], marker='^', color='yellow', edgecolor='black',
                     linewidth=2.0, s=500, zorder=8)
        ax_W.set_xlabel(xlabel_str, fontsize=20)
        ax_W.set_title('W-component [m s$^{-1}$]', fontsize=18)
        ax_W.set_yticklabels([])
        plt.xticks(fontsize=20)
        plt.text(xlim1 + 1, ylim2 - 200, time_label,
                 fontsize=20,
                 bbox=dict(facecolor='white', edgecolor='white', boxstyle='round,pad=0.3'),
                 zorder=10)
        if transition_dists['sub_to_super'] is not None:
            ax_W.axvline(x=transition_dists['sub_to_super'], color='cyan', linewidth=3.5,
                         linestyle='--', label='sub to super', zorder=9)
        if transition_dists['super_to_sub'] is not None:
            ax_W.axvline(x=transition_dists['super_to_sub'], color='orange', linewidth=3.5,
                         linestyle='--', label='super to sub', zorder=9)
        if any(transition_dists.values()):
            ax_W.legend(fontsize=16, loc='upper right')
        plt.ylim(ylim1, ylim2)
        plt.xlim(xlim1, xlim2)

        # ── Diagnostics ─────────────────────────────────────────────────────────
        print(f'Time: {tWRF_strPST}')
        print(f'Fr range: {np.nanmin(fr_profile):.3f} – {np.nanmax(fr_profile):.3f}')
        print(f'Supercritical points: {np.sum(is_super)} / {nx}')
        if transition_dists['sub_to_super'] is not None:
            print(f'Sub to Super transition: {transition_dists["sub_to_super"]:+.2f} km from center')
        else:
            print('Sub to Super transition: not found')
        if transition_dists['super_to_sub'] is not None:
            print(f'Super to Sub transition: {transition_dists["super_to_sub"]:+.2f} km from center')
        else:
            print('Super to Sub transition: not found')

        plt.subplots_adjust(wspace=0.15)

        save_fName = (f'MtnFire_wrf111m_Cross_ang{angle_from_north:.0f}N'
                      f'_Valong_Wcomp_theta_{tWRF_strPST_fName}_PST.jpg')
        fig.savefig(save_fName, bbox_inches='tight', dpi=100)
        fig.clf()
        plt.close()

        del U_cross, V_cross, W_cross, th_cross, pblh_cross, thv_cross
        del Vs_contours, W_contours, th_contours, cbar_V, cbar_W, ax_V, ax_W, fig
        del tWRFstr, tWRF_strPST_fName

    ncfile.close()


Found 38 WRF files
Cross section: center (34.318°N, -118.968°W), angle 35.0° from N
  SW end: (34.2442°N, -119.0306°W)
  NE end: (34.3918°N,  -118.9054°W)
  Along-section wind = U·sin(35°) + V·cos(35°)
  Positive = toward NE end of section

Opened: /glade/derecho/scratch/gduine/mountain_fire/111m/ifire2/wrfout_d04_2024-11-06_05:00:00
Cross-section has 189 grid points
  timestep 0
Time: 20241105 2100
Fr range: 4.265 – 89.709
Supercritical points: 79 / 189
Sub to Super transition: not found
Super to Sub transition: not found

Opened: /glade/derecho/scratch/gduine/mountain_fire/111m/ifire2/wrfout_d04_2024-11-06_06:00:00
  timestep 0
Time: 20241105 2200
Fr range: 0.010 – 25.953
Supercritical points: 187 / 189
Sub to Super transition: +4.78 km from center
Super to Sub transition: +4.62 km from center

Opened: /glade/derecho/scratch/gduine/mountain_fire/111m/ifire2/wrfout_d04_2024-11-06_07:00:00
  timestep 0
Time: 20241105 2300
Fr range: 0.549 – 27.888
Supercritical points: 185 / 189
Sub to 